# Multilingual normalizers\n\n> A registry for conservative, language-aware text normalization.

In [ ]:
# | default_exp multilingual
# | export
from dataclasses import dataclass
from typing import Callable
import re
import unicodedata

import regex

from whisper_normalizer.english import EnglishTextNormalizer
from whisper_normalizer.indic import BengaliNormalizer, HindiNormalizer
from whisper_normalizer.international import FrenchTextNormalizer, SpanishTextNormalizer


class MultilingualTextNormalizer:
    """Conservative, script-safe normalization for languages without bespoke rules."""

    def __call__(self, text: str) -> str:
        text = unicodedata.normalize("NFC", text).lower()
        text = re.sub(r"[<\[][^>\]]*[>\]]", "", text)
        text = re.sub(r"\(([^)]+?)\)", "", text)
        text = text.replace("\u200b", "").replace("\ufeff", "")
        # Preserve Unicode marks: they carry meaning in scripts such as Arabic.
        text = regex.sub(r"[\p{P}\p{S}]+", " ", text)
        return re.sub(r"\s+", " ", text).strip()


@dataclass(frozen=True)
class LanguageSupport:
    """A supported language and the normalization level available for it."""

    code: str
    name: str
    factory: Callable[[], Callable[[str], str]]
    capabilities: tuple[str, ...]


_GENERIC_CAPABILITIES = ("unicode", "punctuation", "whitespace", "case")


LANGUAGE_REGISTRY = {
    "ar": LanguageSupport("ar", "Arabic", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "bn": LanguageSupport("bn", "Bengali", BengaliNormalizer, ("unicode", "script", "tts_mode")),
    "de": LanguageSupport("de", "German", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "en": LanguageSupport("en", "English", EnglishTextNormalizer, ("unicode", "punctuation", "numbers", "spelling")),
    "es": LanguageSupport("es", "Spanish", SpanishTextNormalizer, ("unicode", "punctuation", "numbers", "locale_numbers")),
    "fr": LanguageSupport("fr", "French", FrenchTextNormalizer, ("unicode", "punctuation", "numbers", "locale_numbers")),
    "hi": LanguageSupport("hi", "Hindi", HindiNormalizer, ("unicode", "script", "numbers", "tts_mode")),
    "id": LanguageSupport("id", "Indonesian", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "ja": LanguageSupport("ja", "Japanese", MultilingualTextNormalizer, ("unicode", "punctuation", "whitespace")),
    "ko": LanguageSupport("ko", "Korean", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "pt": LanguageSupport("pt", "Portuguese", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "ru": LanguageSupport("ru", "Russian", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "tr": LanguageSupport("tr", "Turkish", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "ur": LanguageSupport("ur", "Urdu", MultilingualTextNormalizer, _GENERIC_CAPABILITIES),
    "zh": LanguageSupport("zh", "Chinese", MultilingualTextNormalizer, ("unicode", "punctuation", "whitespace")),
}

LANGUAGE_ALIASES = {
    "arabic": "ar",
    "bengali": "bn",
    "chinese": "zh",
    "english": "en",
    "french": "fr",
    "german": "de",
    "hindi": "hi",
    "indonesian": "id",
    "japanese": "ja",
    "korean": "ko",
    "portuguese": "pt",
    "russian": "ru",
    "spanish": "es",
    "turkish": "tr",
    "urdu": "ur",
}


def get_text_normalizer(language: str) -> Callable[[str], str]:
    """Return a fresh normalizer for an ISO 639-1 code or supported language name."""

    key = language.lower().replace("_", "-").split("-", 1)[0]
    key = LANGUAGE_ALIASES.get(key, key)
    try:
        return LANGUAGE_REGISTRY[key].factory()
    except KeyError as error:
        supported = ", ".join(sorted(LANGUAGE_REGISTRY))
        raise ValueError(f"Unsupported language: {language!r}. Supported codes: {supported}") from error


def supported_languages() -> tuple[LanguageSupport, ...]:
    """Return the language registry in ISO-code order."""

    return tuple(LANGUAGE_REGISTRY[code] for code in sorted(LANGUAGE_REGISTRY))


In [ ]:
assert len(supported_languages()) == 15
assert get_text_normalizer("french")("Vingt et un euros.") == "21 euros"
assert get_text_normalizer("es-ES")("El total es 1.234,50.") == "el total es 1234.50"
assert get_text_normalizer("Russian")("Privet, mir!") == "privet mir"
assert get_text_normalizer("zh")("\u4f60\u597d\uff0c\u4e16\u754c\uff01") == "\u4f60\u597d \u4e16\u754c"
try:
    get_text_normalizer("xx")
except ValueError as error:
    assert "Unsupported language" in str(error)
else:
    raise AssertionError("unknown languages must be rejected")
